# Подбор маршрутных Ridge · Windows · RTX 4060 Laptop

Автономный ноутбук для 16 ГБ RAM и 8 ГБ VRAM. Только Ridge: cuML, XGBoost и CatBoost не нужны.
Положите рядом папку data с train_solo_track.parquet и test_solo_track.parquet. В VS Code выберите Jupyter kernel окружения с CuPy.
Сначала выполните проверку CUDA. Не устанавливайте одновременно несколько вариантов CuPy.

По умолчанию MODE='oof', HORIZONS=(1,…,8), BATCH_SIZE=4, DEVICE='cuda'. Все горизонты идут последовательно.
Для каждой пары маршрут × горизонт выбирается alpha по 30 фиксированным кандидатам от 1e-4 до 1e6 (включая 100)
на неделях 30 августа–5 сентября, 6–12 и 13–19 сентября. Это воспроизводимый перебор, без Optuna.
Критерий — средняя недельная WAPE + |Bias|, при нулевом объёме недели — MAE с отметкой.
Train — пять стартов 09:30–11:30 по всей доступной истории, метки строго до соответствующей недели.
После подбора alpha фиксируется и строятся OOF-2, OOF-3, OOF-4 (27 сентября–17 октября).

Прогресс сохраняется после каждого пакета. При остановке повторите Run All с теми же путями и окружением:
готовые пакеты и горизонты будут пропущены, временные признаки могут строиться заново.
Не удаляйте output; cache содержит только временные файлы. Перед сном компьютера остановите расчёт;
для длительного запуска подключите питание и отключите автоматический сон.

После OOF перенесите архив ridge_tuned_*_export.zip в Kaggle Dataset и используйте новые stacking_tuned_part_*.ipynb.
Старые четыре Kaggle-ноутбука продолжают работу без изменений. Новые Ridge добавляются отдельной колонкой.
Позже MODE='evaluate' построит test, MODE='submit' — extra и full; alpha не переизбирается.
После каждого этапа обновите Kaggle Dataset новым архивом целиком, не подключайте несколько версий одновременно.
Выбирайте конфигурацию стекинга до просмотра test. Локальные поздние этапы могут считаться заранее, но не используйте их метрики для выбора.

In [ ]:
PROFILE = 'five_starts'
MODE = 'oof'
ROLE = 'worker'
DEVICE = 'cuda'
HORIZONS = (1,2,3,4,5,6,7,8)
BATCH_SIZE = 4
INPUT_DIR = './data'
OUTPUT_DIR = './ridge_output'
CACHE_DIR = './ridge_cache'
BASE_CODE_ID = 'c9075a4e3ad5ce73e0de4f84bc82fa50836df388d8a65c54e9a6b0db72dc601b'
TUNED_CODE_ID = 'b468280b115a0cd0dc242ecf188e0b712458f57fb1fb3a1a2b3b8f2a19ba35f3'

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

START = pd.Timestamp('2025-09-20')
TEST_START = pd.Timestamp('2025-10-18')
DAY = pd.Timedelta(days=1)
STEP = pd.Timedelta(minutes=30)
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
TRAIN_MINUTES = (570, 600, 630, 660, 690)
REGIMES = {'five_starts':('five_starts',), 'recent75':('recent_all',)}[PROFILE]
STACK_WEEKS = (2, 3, 4)
VALIDATION_WEEKS = STACK_WEEKS[1:]
FAMILIES = ('rf', 'light_gbm', 'catboost', 'xgb', 'ridge')
MODEL_NAMES = [f'{f}_{v}' for f in FAMILIES for v in ('direct', 'residual')]
KEY = ['cutoff', 'route_id', 'horizon']

TARGET_SCALE = 100000
ETA_CONTEXT = None
THREADS = min(4, os.cpu_count() or 1)

def file_digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def locate(name):
    hits = list(INPUT.rglob(name))
    if len(hits) != 1:
        raise ValueError(f'Ожидается один {name}, найдено {hits}')
    return hits[0]

def initialize():
    global INPUT, OUTPUT, ROOT, TRAIN_PATH, TEST_PATH, RUN_ID, CHECKPOINTS
    INPUT = Path(INPUT_DIR)
    TRAIN_PATH = locate('train_solo_track.parquet')
    TEST_PATH = locate('test_solo_track.parquet')
    fingerprints = [file_digest(TRAIN_PATH),file_digest(TEST_PATH)]
    identity = [CODE_ID,PROFILE,str(ROUTE_ALPHA),*fingerprints]
    if not np.isfinite(ROUTE_ALPHA) or ROUTE_ALPHA<=0:
        raise ValueError('ROUTE_ALPHA должен быть положительным')
    RUN_ID = hashlib.sha256('|'.join(identity).encode()).hexdigest()[:24]
    OUTPUT = Path(OUTPUT_DIR)
    ROOT = OUTPUT / f'stack_{RUN_ID}'
    ROOT.mkdir(parents=True, exist_ok=True)
    CHECKPOINTS = [p for p in INPUT.rglob(ROOT.name) if p.is_dir() and p.resolve() != ROOT.resolve()]
    print(f'Конфигурация {RUN_ID}, часть {PART}, режим {MODE}, роль {ROLE}')

def profile_path(rel):
    return str(rel)

def existing(rel):
    rel = profile_path(rel)
    dest = ROOT / rel
    if dest.is_file():
        return dest
    hits = [base / rel for base in CHECKPOINTS if (base / rel).is_file()]
    if not hits:
        return None
    if len(hits) > 1 and len({file_digest(p) for p in hits}) != 1:
        raise ValueError(f'Несогласованные копии {rel}: {hits}')
    dest.parent.mkdir(parents=True, exist_ok=True)
    temp = dest.with_suffix('.copying')
    shutil.copyfile(hits[0], temp)
    temp.replace(dest)
    return dest

def require(rel):
    path = existing(rel)
    if path is None:
        raise FileNotFoundError(f'Подключите результат {rel} из запуска {RUN_ID}')
    return path

def save(rel, value):
    path = ROOT / profile_path(rel)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix('.partial')
    if isinstance(value, pd.DataFrame):
        value.to_parquet(temp, index=False)
    else:
        temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)

def load_json(rel):
    return json.loads(require(rel).read_text())

def train_indices(h, boundary, regime):
    possible = np.flatnonzero(times[:len(times)-h] + h * STEP < boundary)
    days = times[possible].normalize()
    unique = days.unique()
    early = days.isin(unique[:len(unique)//4])
    five = np.isin(times[possible].hour * 60 + times[possible].minute, TRAIN_MINUTES)
    masks = {'five_starts': five, 'mixed_starts': ~early | five, 'recent_all': ~early}
    return possible[masks[regime]]

def query_indices(boundary, days):
    if days == 0:
        return np.array([len(times)-1])
    grid = pd.date_range(boundary + pd.Timedelta(hours=10, minutes=30), periods=days, freq='D')
    result = times.get_indexer(grid)
    if (result < 0).any():
        raise ValueError(f'Нет стартов для {grid}')
    return result

def metric(y, p):
    y = np.asarray(y, dtype=np.float64)
    error = np.asarray(p, dtype=np.float64) - y
    if not len(y) or not np.isfinite(error).all():
        raise ValueError('Пустая выборка или нечисловой прогноз')
    volume = float(y.sum())
    return totals(len(y), volume, float(np.abs(error).sum()), float(error.sum()))

def totals(n, volume, absolute, signed):
    score = (absolute+abs(signed))/volume if volume else None
    return dict(rows=int(n), volume=volume, absolute=absolute, signed=signed,
                WAPE=absolute/volume if volume else None, relative_bias=signed/volume if volume else None,
                score=score, **{'WAPE + |Relative Bias|':score})

def combine(rows):
    return totals(*(sum(row[k] for row in rows) for k in ('rows','volume','absolute','signed')))

def task_path(h, stage, regime, name):
    return f'h{h}/{stage}/{regime}/{name}.parquet'

def complete(rel):
    return existing(rel + '.json') is not None and existing(rel) is not None

def result_frame(h, origins):
    frame = pd.DataFrame(dict(cutoff=times[origins].repeat(len(routes)), route_id=np.tile(routes, len(origins))))
    frame['horizon'] = h
    frame['timestamp'] = frame.cutoff + h * STEP
    if origins[-1] + h < len(times):
        frame['y'] = target[origins+h].ravel()
    return frame

def finish_fit(rel, frame, pred, train_row, boundary, fit_through, seconds, device):
    if not np.isfinite(pred).all() or np.min(pred) < 0:
        raise ValueError(f'Некорректный прогноз {rel}')
    rows = {'train':train_row}
    if 'y' in frame:
        rows['test' if '/test/' in rel else 'held_out'] = metric(frame.y, pred)
    save(rel, frame.assign(prediction=pred))
    save(rel+'.json',dict(run_id=RUN_ID, boundary=str(boundary), fit_through=str(fit_through),
                          seconds=seconds, backend=device, metrics=rows))
    display(pd.DataFrame(rows).T.assign(task=rel))
    show_eta()

def show_eta():
    if not ETA_CONTEXT:
        return
    known = ETA_CONTEXT['completed']
    oof = [v for (h,s),v in known.items() if s.startswith('oof')]
    if not oof:
        return
    current = ETA_CONTEXT['current']
    remaining = 0.
    for h,stage in ETA_CONTEXT['pending']:
        if (h,stage) in known:
            continue
        similar = [v for (other,s),v in known.items() if s==stage]
        if not similar:
            similar = [v for (other,s),v in known.items() if s[:3]==stage[:3]]
        estimate = np.mean(similar or oof)
        if (h,stage)==current:
            estimate = max(0.,estimate-(time.monotonic()-ETA_CONTEXT['start']))
        remaining += estimate
    remaining += ETA_CONTEXT['future_features']*ETA_CONTEXT['features']
    print(f'ETA части по измеренным этапам: {remaining/3600:.2f} ч')

def stage_spec(stage):
    if stage.startswith('oof'):
        return START + (int(stage[-1])-1)*7*DAY, 7
    if stage == 'test':
        return TEST_START, 14
    if stage == 'extra':
        return TEST_START + 7*DAY, 7
    if stage == 'full':
        return times[-1] + STEP, 0
    raise ValueError(stage)

def device_setup():
    global xp, cp, cupyx, BACKEND, LGB_DEVICE
    try:
        import cupy as cp
        import cupyx
        from cuml.ensemble import RandomForestRegressor
        device_name = cp.cuda.runtime.getDeviceProperties(0)['name']
    except (ImportError,RuntimeError) as error:
        raise RuntimeError('Для расчёта баз включите NVIDIA GPU и подключите CuPy и cuML, совместимые с CUDA сессии.') from error
    xp, BACKEND = cp, 'cuda'
    cp.linalg.solve(cp.eye(2),cp.ones(2))
    LGB_DEVICE = 'cpu'
    for candidate in ['cuda','gpu']:
        code = f"""import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
r = np.tile(np.arange({len(routes)}),32)
rng = np.random.default_rng(42)
data = pd.DataFrame(dict(route_id=pd.Categorical(r),volume=rng.random(len(r))))
model = LGBMRegressor(device_type='{candidate}',n_estimators=2,max_cat_to_onehot=1,verbosity=-1,n_jobs={THREADS})
model.fit(data,r%17+data.volume,categorical_feature=['route_id'])
"""
        probe = subprocess.run([sys.executable,'-c',code],capture_output=True,text=True)
        if probe.returncode == 0:
            LGB_DEVICE = candidate
            break
        print(candidate, (probe.stderr or probe.stdout)[-1200:])
    print('GPU:',device_name,'LightGBM:',LGB_DEVICE)
    from importlib.metadata import version
    save(f'part{PART}/{MODE}_devices.json',dict(gpu=str(device_name),lightgbm_device=LGB_DEVICE,
         packages={k:version(k) for k in ('numpy','pandas','xgboost','lightgbm','catboost','scikit-learn')},
         cupy=cp.__version__,cuml=__import__('cuml').__version__))

In [ ]:
from contextlib import nullcontext
import platform
import zipfile
import psutil
from importlib.metadata import version

TUNING_STARTS = tuple(pd.Timestamp(s) for s in ('2025-08-30','2025-09-06','2025-09-13'))
ALPHAS = np.sort(np.unique(np.r_[np.geomspace(1e-4,1e6,29),100.]))
TUNED_COLUMN = 'five_starts__route_ridge_tuned'

def local_json(path,value):
    path.parent.mkdir(parents=True,exist_ok=True)
    temp = path.with_suffix(path.suffix+'.partial')
    temp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    temp.replace(path)

def local_parquet(path,frame):
    path.parent.mkdir(parents=True,exist_ok=True)
    temp = path.with_suffix('.partial')
    frame.to_parquet(temp,index=False)
    temp.replace(path)

def local_npz(path,**arrays):
    path.parent.mkdir(parents=True,exist_ok=True)
    temp = path.with_suffix('.partial')
    with temp.open('wb') as stream:
        np.savez_compressed(stream,**arrays)
    temp.replace(path)

def local_initialize():
    global INPUT,TRAIN_PATH,TEST_PATH,LOCAL_ROOT,MANIFEST,xp,cp,cupyx,BACKEND
    if MODE not in ('oof','evaluate','submit') or not HORIZONS or any(h not in range(1,9) for h in HORIZONS):
        raise ValueError('MODE: oof/evaluate/submit; HORIZONS: 1–8')
    if BATCH_SIZE not in range(1,17):
        raise ValueError('BATCH_SIZE должен быть от 1 до 16, рекомендуется 4')
    INPUT = Path(INPUT_DIR).resolve()
    TRAIN_PATH,TEST_PATH = locate('train_solo_track.parquet'),locate('test_solo_track.parquet')
    packages = {name:version(name) for name in ('numpy','pandas','scikit-learn','pyarrow')}
    fingerprints = [file_digest(TRAIN_PATH),file_digest(TEST_PATH)]
    identity = dict(code_id=TUNED_CODE_ID,base_code_id=BASE_CODE_ID,profile=PROFILE,
                    fingerprints=fingerprints,alphas=ALPHAS.tolist(),packages=packages)
    run_id = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:24]
    LOCAL_ROOT = Path(OUTPUT_DIR).resolve()/f'ridge_tuned_{run_id}'
    LOCAL_ROOT.mkdir(parents=True,exist_ok=True)
    MANIFEST = dict(**identity,run_id=run_id,column=TUNED_COLUMN,tuning_end='2025-09-20',
                    tuning_starts=[str(s) for s in TUNING_STARTS],oof_weeks=list(STACK_WEEKS))
    local_json(LOCAL_ROOT/'manifest.json',MANIFEST)
    cp,cupyx = None,None
    if DEVICE=='cuda':
        try:
            import cupy as cp
            import cupyx
            xp,BACKEND = cp,'cuda'
            cp.linalg.solve(cp.eye(2),cp.ones(2))
            cp.linalg.eigh(cp.eye(3)[None,:,:])
            cp.cuda.runtime.deviceSynchronize()
        except Exception as error:
            raise RuntimeError('CUDA не готова. Проверьте CuPy, драйвер и выбранное окружение Jupyter. См. инструкцию Windows.') from error
        gpu = str(cp.cuda.runtime.getDeviceProperties(0)['name'])
        free,total = cp.cuda.runtime.memGetInfo()
        print(f'GPU: {gpu}; свободно {free/2**30:.2f}/{total/2**30:.2f} ГиБ',flush=True)
    elif DEVICE=='cpu':
        xp,BACKEND = np,'cpu'
    else:
        raise ValueError('DEVICE: cuda или cpu')
    local_json(LOCAL_ROOT/'environment.json',dict(device=DEVICE,platform=platform.platform(),
               packages=packages,cupy=cp.__version__ if cp else None))
    print(f'RAM свободно: {psutil.virtual_memory().available/2**30:.2f} ГиБ; результаты: {LOCAL_ROOT}',flush=True)

def release_device():
    gc.collect()
    if BACKEND=='cuda':
        cp.get_default_memory_pool().free_all_blocks()

def close_horizon(data):
    for name in ('frame','corr'):
        array = getattr(data,name,None)
        if array is not None and hasattr(array,'_mmap'):
            array.flush()
            array._mmap.close()
            setattr(data,name,None)

def timed_log(event,started,**details):
    synchronize()
    row = dict(event=event,seconds=time.monotonic()-started,**details)
    with (LOCAL_ROOT/'timings.jsonl').open('a',encoding='utf-8') as stream:
        stream.write(json.dumps(row,ensure_ascii=False)+'\n')
    print(row,flush=True)

def tune_items(items):
    batch = DeviceBatch(items,spectral=True)
    scores = np.empty((len(items),len(ALPHAS)))
    zero = np.array([item['query_y'].sum()==0 for item in items])
    for j,alpha in enumerate(ALPHAS):
        coef = batch.coefficients(np.full(len(items),alpha),True)
        predictions = to_host(xp.maximum((batch.q@coef[:,:,None])[:,:,0]+batch.y_mean[:,None],0.))
        for r,item in enumerate(items):
            actual = item['query_y']
            error = predictions[r,:len(actual)]-actual
            volume = actual.sum()
            scores[r,j] = (np.abs(error).sum()+abs(error.sum()))/volume if volume>0 else np.abs(error).mean()
    if not np.isfinite(scores).all():
        raise ValueError('Нечисловые оценки подбора')
    return scores,zero

def fit_items(items,alphas,reference=False):
    batch = DeviceBatch(items,spectral=False)
    coef = batch.coefficients(alphas,False)
    metrics,predictions = batch.metrics(coef)
    if reference:
        for r,item in enumerate(items[:2]):
            expected = np.maximum(0.,Ridge(alpha=alphas[r],solver='cholesky').fit(item['x'],item['y']).predict(item['query_x']))
            if not np.allclose(expected,predictions[r,:len(expected)],rtol=1e-6,atol=1e-7*max(1.,np.max(np.abs(item['y'])))):
                raise ValueError('Ridge расходится с эталоном sklearn')
    return predictions,metrics

def retry_compute(items,alphas=None,reference=False):
    try:
        return tune_items(items) if alphas is None else fit_items(items,alphas,reference)
    except Exception as error:
        oom = isinstance(error,MemoryError) or (BACKEND=='cuda' and isinstance(error,cp.cuda.memory.OutOfMemoryError))
        if not oom or len(items)==1:
            raise
    release_device()
    middle = len(items)//2
    left = retry_compute(items[:middle],None if alphas is None else alphas[:middle],reference)
    right = retry_compute(items[middle:],None if alphas is None else alphas[middle:],reference)
    if alphas is None:
        return np.concatenate([left[0],right[0]]),np.concatenate([left[1],right[1]])
    return np.concatenate([left[0],right[0]]),{k:left[1][k]+right[1][k] for k in left[1]}

def tuning_fold(data,fold,boundary):
    origins = train_indices(data.h,boundary,'five_starts')
    query = query_indices(boundary,7)
    folder = LOCAL_ROOT/f'h{data.h}/tuning/fold{fold}'
    groups = [(start,min(start+BATCH_SIZE,len(routes))) for start in range(0,len(routes),BATCH_SIZE)]
    if any(not (folder/f'{start}_{end}.npz').is_file() for start,end in groups):
        started = time.monotonic()
        data.prepare(boundary,int(query[-1])+1)
        timed_log('tuning_lags',started,horizon=data.h,fold=fold)
    scores,zero = [],[]
    for start,end in groups:
        path = folder/f'{start}_{end}.npz'
        if not path.is_file():
            started = time.monotonic()
            items = [route_item(data,origins,query,r,'tuning') for r in range(start,end)]
            result,flags = retry_compute(items)
            local_npz(path,scores=result,zero_volume=flags,route_ids=routes[start:end])
            del items
            release_device()
            timed_log('tuning_batch',started,horizon=data.h,fold=fold,completed=end,total=len(routes))
        with np.load(path,allow_pickle=False) as batch:
            if not np.array_equal(batch['route_ids'],routes[start:end]) or batch['scores'].shape!=(end-start,len(ALPHAS)):
                raise ValueError(f'Некорректный checkpoint {path}')
            scores.append(batch['scores'].copy())
            zero.append(batch['zero_volume'].copy())
    return np.concatenate(scores),np.concatenate(zero)

def tuned_alphas(data):
    path = LOCAL_ROOT/f'h{data.h}/alphas.parquet'
    if path.is_file():
        table = pd.read_parquet(path)
        if not np.array_equal(table.route_id,routes):
            raise ValueError('Несовпадение маршрутов в alpha')
        return table.alpha.to_numpy()
    results,flags = [],[]
    for fold,boundary in enumerate(TUNING_STARTS,1):
        result,zero = tuning_fold(data,fold,boundary)
        results.append(result)
        flags.append(zero)
    scores = np.stack(results).mean(axis=0)
    selected = scores.argmin(axis=1)
    alpha = ALPHAS[selected]
    table = pd.DataFrame(dict(route_id=routes,horizon=data.h,alpha=alpha,
                 mean_tuning_score=scores[np.arange(len(routes)),selected],zero_volume_windows=np.stack(flags).sum(axis=0)))
    local_parquet(path,table)
    local_npz(path.with_suffix('.npz'),scores=scores,alphas=ALPHAS,route_ids=routes)
    display(table.alpha.describe())
    return alpha

def prediction_path(h,stage):
    return LOCAL_ROOT/f'predictions/h{h}/{stage}.parquet'

def predict_stage(data,stage,alphas):
    path = prediction_path(data.h,stage)
    info_path = path.with_suffix('.json')
    if path.is_file() and info_path.is_file():
        return
    boundary,days = stage_spec(stage)
    origins = train_indices(data.h,boundary,'five_starts')
    query = query_indices(boundary,days)
    if not (times[origins]+data.h*STEP < boundary).all():
        raise ValueError('Утечка целевых меток')
    folder = LOCAL_ROOT/f'h{data.h}/batches/{stage}'
    groups = [(start,min(start+BATCH_SIZE,len(routes))) for start in range(0,len(routes),BATCH_SIZE)]
    if any(not (folder/f'{start}_{end}.json').is_file() for start,end in groups):
        started = time.monotonic()
        data.prepare(boundary,int(query[-1])+1)
        timed_log('prediction_lags',started,horizon=data.h,stage=stage)
    predictions,train_rows = [],[]
    for start,end in groups:
        checkpoint = folder/f'{start}_{end}.json'
        if not checkpoint.is_file():
            started = time.monotonic()
            items = [route_item(data,origins,query,r,stage) for r in range(start,end)]
            out,metrics = retry_compute(items,alphas[start:end],start==0)
            rows = {key:combine([totals(v['n'],v['actual_sum'],v['absolute_error'],v['signed_error']) for v in val]) for key,val in metrics.items()}
            local_json(checkpoint,dict(predictions=out.tolist(),metrics=rows,route_ids=routes[start:end].tolist()))
            del items
            release_device()
            timed_log('prediction_batch',started,horizon=data.h,stage=stage,completed=end,total=len(routes))
        batch = json.loads(checkpoint.read_text(encoding='utf-8'))
        if not np.array_equal(batch['route_ids'],routes[start:end]):
            raise ValueError('Несовпадение маршрутов checkpoint')
        predictions.append(np.asarray(batch['predictions']))
        train_rows.append(batch['metrics']['train'])
    pred = np.concatenate(predictions).T.ravel()
    frame = result_frame(data.h,query).assign(prediction=pred)
    if not np.isfinite(pred).all() or (pred<0).any() or frame.duplicated(KEY).any():
        raise ValueError('Некорректный прогноз')
    rows = dict(train=combine(train_rows))
    if 'y' in frame:
        rows['held_out'] = metric(frame.y,pred)
    local_parquet(path,frame)
    local_json(info_path,dict(run_id=MANIFEST['run_id'],base_code_id=BASE_CODE_ID,boundary=str(boundary),
               fit_through=str(times[origins[-1]]+data.h*STEP),tuning_end='2025-09-20',metrics=rows))
    display(pd.DataFrame(rows).T.assign(horizon=data.h,stage=stage))

def export_results():
    destination = LOCAL_ROOT.parent/f'{LOCAL_ROOT.name}_export.zip'
    temp = destination.with_suffix('.partial')
    files = [LOCAL_ROOT/'manifest.json',LOCAL_ROOT/'environment.json']
    files += list(LOCAL_ROOT.glob('h*/alphas.parquet'))+list(LOCAL_ROOT.glob('predictions/h*/*'))
    with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for path in files:
            archive.write(path,arcname=str(Path(LOCAL_ROOT.name)/path.relative_to(LOCAL_ROOT)))
    temp.replace(destination)
    print(f'Экспорт для Kaggle: {destination}',flush=True)

def run_local():
    from threadpoolctl import threadpool_limits
    stages = [f'oof{i}' for i in STACK_WEEKS] if MODE=='oof' else ['test'] if MODE=='evaluate' else ['extra','full']
    with threadpool_limits(limits=2):
        for h in HORIZONS:
            pending = [stage for stage in stages if not (prediction_path(h,stage).is_file() and prediction_path(h,stage).with_suffix('.json').is_file())]
            if not pending:
                continue
            if MODE!='oof' and not (LOCAL_ROOT/f'h{h}/alphas.parquet').is_file():
                raise ValueError(f'Сначала выполните oof для h{h}, чтобы зафиксировать alpha')
            stop = max(query_indices(*stage_spec(stage))[-1] for stage in pending)+1
            scratch = Path(CACHE_DIR).resolve()
            scratch.mkdir(parents=True,exist_ok=True)
            if psutil.virtual_memory().available<2*2**30:
                raise MemoryError('Перед расчётом горизонта нужно освободить хотя бы 2 ГиБ RAM')
            with TemporaryDirectory(dir=scratch) as folder:
                started = time.monotonic()
                data = HorizonData(folder,h,int(stop))
                try:
                    timed_log('horizon_features',started,horizon=h)
                    alphas = tuned_alphas(data)
                    for stage in pending:
                        predict_stage(data,stage,alphas)
                finally:
                    close_horizon(data)
                    del data
                    release_device()
            export_results()
    export_results()

In [ ]:
local_initialize()

In [ ]:
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
test = pd.read_parquet(TEST_PATH, columns=['id', 'route_id', 'timestamp'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
if not (test.id.is_unique and (not test.duplicated(['route_id', 'timestamp']).any())):
    raise ValueError('Повторы идентификаторов или ключей теста')
end = times[-1]
if not (end.hour, end.minute) == (10, 30):
    raise ValueError('Последняя точка истории должна быть в 10:30')
expected = pd.MultiIndex.from_product([routes, [end + h * STEP for h in range(1, 9)]], names=['route_id', 'timestamp'])
if not expected.difference(pd.MultiIndex.from_frame(test[['route_id', 'timestamp']])).empty:
    raise ValueError('Тест не покрывает все ожидаемые маршруты и горизонты')
if not len(test) == len(expected):
    raise ValueError('Число строк теста отличается от ожидаемого')
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')

In [ ]:
target = values[TARGET]
if MODE != 'assemble' and ROLE != 'collect':
    target = values[TARGET]

    def past(array, positions):
        positions = np.asarray(positions, dtype=np.int64)
        valid = (positions >= 0) & (positions < len(array))
        result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
        result[~valid] = np.nan
        return result

    def average_available(matrices):
        total = np.zeros_like(matrices[0], dtype=np.float32)
        count = np.zeros_like(total)
        for matrix in matrices:
            valid = np.isfinite(matrix)
            total += np.where(valid, matrix, 0)
            count += valid
        return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

    def rolling(array, steps):
        return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
    means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
    level24 = rolling(target, 48)
    positions = np.arange(len(times))
    extra_signals = {}
    for name in STATUSES[:3]:
        extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
    extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
    extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
    for name in STATUSES[3:]:
        extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
    extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
    extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
    extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
    hub_signal = extra_signals['status_6_diff_hourly']
    extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
    recent = means[TARGET, 4]
    usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
    extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

    def offset_name(base, offset):
        if offset == 0:
            return base
        return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

    def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
        for offset in offsets:
            common[offset_name(base, offset)] = past(array, anchor + offset)

    def common_features(origins):
        common = {}
        local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
        for lag in local_target_lags:
            common[f'target_back_{lag}'] = past(target, origins - lag)
        causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
        for name, array in values.items():
            for steps in (4, 12):
                add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
            if name in STATUSES:
                common[f'{name}_latest'] = array[origins]
                for lag in range(1, 5):
                    common[f'{name}_back_{lag}'] = past(array, origins - lag)
                if name in STATUSES[:3]:
                    change = array - past(array, positions - 2)
                    add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
        for name, array in extra_signals.items():
            add_window(common, name, array, origins, causal_offsets)
        for lag in (48, 96):
            add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
        add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
        add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
        return common
    CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

    for name, series in values.items():
        network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
        for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
            extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)


    def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
        grid = pd.DatetimeIndex(timestamps)
        route_ids = np.asarray(route_ids)
        target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
        if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
            raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
        if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
            raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
        if len(np.unique(route_ids)) != len(route_ids):
            raise ValueError('Идентификаторы маршрутов повторяются')
        steps = float(max_lookback_hours) * 2
        if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
            raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
        if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
            raise ValueError('min_pairs должен быть целым числом >=3')
        if not 0 <= min_correlation <= 1 or route_block_size < 1:
            raise ValueError('Некорректные параметры выбора корреляции')
        max_lag = int(steps)
        stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
        anchors = np.arange(max_lag + 2, stop)
        best_corr = np.full(len(route_ids), np.nan)
        best_lag = np.full(len(route_ids), np.nan)
        best_pairs = np.zeros(len(route_ids), dtype=np.int32)
        for start in range(0, len(route_ids), route_block_size):
            finish = min(start + route_block_size, len(route_ids))
            x = target_values[anchors, start:finish].astype(np.float64)
            for lag in range(2, max_lag + 1):
                z = status6_diff[anchors - lag, start:finish].astype(np.float64)
                valid = np.isfinite(x) & np.isfinite(z)
                count = valid.sum(axis=0)
                xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
                divisor = np.maximum(count, 1)
                sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
                covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
                vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
                vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
                denominator = np.sqrt(vx * vz)
                corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
                corr = np.clip(corr, -1, 1)
                previous = best_corr[start:finish]
                better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
                best_corr[start:finish][better] = corr[better]
                best_lag[start:finish][better] = lag
                best_pairs[start:finish][better] = count[better]
        accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
        return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

    def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
        if h not in range(1, 9) or chunk_rows < 1:
            raise ValueError('Некорректный горизонт или размер блока')
        extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
        names = CORRELATION_COLUMNS + extra_columns
        result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
        if lag_table is None:
            names = [prefix + name for name in names]
            return pd.concat([frame.drop(columns=names, errors='ignore'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
        by_route = lag_table.set_index('route_id').reindex(routes)
        route_index = pd.Index(routes)
        lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
        for start in range(0, len(frame), chunk_rows):
            finish = min(start + chunk_rows, len(frame))
            block = frame.iloc[start:finish]
            origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
            route = route_index.get_indexer(block.route_id)
            if (origin < 0).any() or (route < 0).any():
                raise ValueError('Строка не сопоставлена с историей')
            selected = lag[route]
            for j, name in enumerate(CORRELATION_COLUMNS[:4]):
                result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
            for offset in WINDOW_OFFSETS:
                source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
                available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
                signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
                available &= np.isfinite(signal)
                result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
                result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
        names = [prefix + name for name in names]
        frame = frame.drop(columns=names, errors='ignore')
        return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
    def remove_seasonality(series):
        typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
        return series - typical

    residual_target = remove_seasonality(target)
    residual_signal = remove_seasonality(hub_signal)

    def fit_lags(boundary):
        tables = []
        for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
            table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
            tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
        return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

    def add_correlation_features(frame,h,lag_table):
        frame = correlation_block(frame,h,lag_table,hub_signal,'')
        if lag_table is None:
            return correlation_block(frame,h,None,residual_signal,'residual_')
        residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
        residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
        return correlation_block(frame,h,residual_table,residual_signal,'residual_')



    def horizon_dataset(h, origins):
        slot = origins + h
        dates = times[origins] + h * STEP
        common = common_features(origins)
        frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
        calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
        calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
        calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
        for name, array in calendar.items():
            frame[name] = np.repeat(array, len(routes))
        frame = add_correlation_features(frame, h, None)
        seasonal_columns = {}
        for offset in WINDOW_OFFSETS:
            for day in (1, 2, 7):
                seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
            for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
                seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
        for base in ('slot_daily7','slot_weekly4'):
            window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
            valid = np.isfinite(window)
            maximum = np.where(valid, window, -np.inf).max(axis=1)
            minimum = np.where(valid, window, np.inf).min(axis=1)
            peak = valid & (window == maximum[:,None])
            count = peak.sum(axis=1)
            shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
            seasonal_columns[base+'_peak_minutes'] = shift * 30
            seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
            seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
        frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
        profile = np.zeros((len(origins), len(routes)))
        counts = np.zeros_like(profile)
        for day in range(1, 29):
            level = past(level24, origins - day * 48)
            adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
            profile += np.nan_to_num(adjusted)
            counts += np.isfinite(adjusted)
        frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
        if h == 1:
            lower = np.zeros_like(target[origins])
            upper = past(target, origins - 2)
            for position in (origins - 1, origins):
                total = past(target, position)
                lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
            seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
            frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
        else:
            frame['overlap'] = np.nan
        raw_target_sources = []
        for lag in range(9):
            raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
        for lag in (48, 96):
            for offset in WINDOW_OFFSETS:
                raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
        for day in (1, 2, 7):
            for offset in WINDOW_OFFSETS:
                raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
        seen, redundant = (set(), [])
        for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
            if source in seen:
                redundant.append(name)
            else:
                seen.add(source)
        frame = frame.drop(columns=redundant)
        numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
        return frame.astype({name: 'float32' for name in numeric}).copy()

In [ ]:
class HorizonData:
    def __init__(self, folder, h, stop=None):
        self.h = h
        self.stop = len(times) if stop is None else stop
        self.folder = Path(folder)
        self.corr_names = None
        self.corr = None
        self.frame = None
        self.baseline = np.empty((self.stop,len(routes)),np.float32)
        self.origins = np.arange(self.stop)
        if PROFILE=='five_starts':
            grid = times[:self.stop]
            self.origins = self.origins[np.isin(grid.hour*60+grid.minute,TRAIN_MINUTES)]
        self.slots = np.full(self.stop,-1,dtype=int)
        self.slots[self.origins] = np.arange(len(self.origins))
        for start in range(0,len(self.origins),8):
            idx = self.origins[start:start+8]
            frame = horizon_dataset(h,idx)
            if self.frame is None:
                self.cols = [c for c in frame if c not in ('cutoff','timestamp','route_id','profile28','overlap')
                             and not (c.startswith(('selected_status6','status6_lag_valid','lagged_status6','residual_')))]
                self.frame = np.memmap(self.folder/'static.bin', mode='w+', dtype='float32',
                                       shape=(len(self.origins),len(routes),len(self.cols)))
            self.frame[self.slots[idx]] = frame[self.cols].to_numpy().reshape(len(idx),len(routes),-1)
            self.baseline[idx] = frame.profile28.to_numpy().reshape(len(idx),len(routes))
        self.frame.flush()

    def prepare(self, boundary, stop):
        table = fit_lags(boundary)
        origins = self.origins[self.origins<stop]
        for start in range(0,len(origins),16):
            idx = origins[start:start+16]
            keys = result_frame(self.h,idx)[['cutoff','route_id']]
            corr = add_correlation_features(keys,self.h,table).drop(columns=['cutoff','route_id'])
            if self.corr is None:
                self.corr_names = corr.columns.tolist()
                self.corr = np.memmap(self.folder/'correlations.bin',mode='w+',dtype='float32',
                                      shape=(len(self.origins),len(routes),len(self.corr_names)))
            self.corr[self.slots[idx]] = corr.to_numpy().reshape(len(idx),len(routes),-1)

    def matrix(self, origins, route=None):
        slots = self.slots[origins]
        if (slots<0).any():
            raise ValueError('Запрошен старт вне выбранного профиля')
        if route is None:
            a,b = self.frame[slots], self.corr[slots]
        else:
            a,b = self.frame[slots,route], self.corr[slots,route]
        return np.concatenate([a,b],axis=-1).reshape(-1,len(self.cols)+len(self.corr_names))

In [ ]:
from contextlib import nullcontext
def to_host(value):
    return cp.asnumpy(value) if BACKEND == 'cuda' else np.asarray(value)

def synchronize():
    if BACKEND == 'cuda':
        cp.cuda.get_current_stream().synchronize()

def linalg_guard():
    return cupyx.errstate(linalg='raise') if BACKEND == 'cuda' else nullcontext()

def totals_to_metrics(values):
    count, volume, predicted, absolute, signed = [float(value) for value in values]
    if count < 1 or not np.isfinite(values).all():
        raise ValueError('Нечисловые или пустые пакетные метрики')
    return dict(n=int(round(count)), actual_sum=volume, prediction_sum=predicted, absolute_error=absolute,
                signed_error=signed, WAPE=absolute / volume if volume > 0 else None,
                abs_relative_bias=abs(signed) / volume if volume > 0 else None,
                score=(absolute + abs(signed)) / volume if volume > 0 else None, MAE=absolute / count)

class DeviceBatch:
    def __init__(self, items, spectral):
        self.items = items
        self.size = len(items)
        self.width = max(item['x'].shape[1] for item in items)
        rows = max(len(item['y']) for item in items)
        query_rows = max(len(item['query_y']) for item in items)
        self.x = xp.zeros((self.size, rows, self.width), dtype=xp.float64)
        self.q = xp.zeros((self.size, query_rows, self.width), dtype=xp.float64)
        self.y = xp.zeros((self.size, rows), dtype=xp.float64)
        self.qy = xp.zeros((self.size, query_rows), dtype=xp.float64)
        self.mask = xp.zeros((self.size, rows), dtype=xp.bool_)
        self.original_mask = xp.zeros((self.size, rows), dtype=xp.bool_)
        self.query_mask = xp.zeros((self.size, query_rows), dtype=xp.bool_)
        self.y_mean = xp.asarray([item['y_mean'] for item in items], dtype=xp.float64)
        for index, item in enumerate(items):
            n, p = item['x'].shape
            qn = len(item['query_y'])
            self.x[index, :n, :p] = xp.asarray(item['x'])
            self.q[index, :qn, :p] = xp.asarray(item['query_x'])
            self.y[index, :n] = xp.asarray(item['y'])
            self.qy[index, :qn] = xp.asarray(item['query_y'])
            self.mask[index, :n] = True
            self.original_mask[index, :n] = xp.asarray(item['original_train'])
            self.query_mask[index, :qn] = True
        transposed = self.x.swapaxes(-1, -2)
        self.gram = transposed @ self.x
        centered_y = xp.where(self.mask, self.y - self.y_mean[:, None], 0.0)
        self.rhs = (transposed @ centered_y[:, :, None])[:, :, 0]
        if spectral:
            with linalg_guard():
                values, self.vectors = xp.linalg.eigh(self.gram)
            self.eigenvalues = xp.maximum(values, 0.0)
            self.projected = (self.vectors.swapaxes(-1, -2) @ self.rhs[:, :, None])[:, :, 0]
        synchronize()

    def coefficients(self, alphas, spectral):
        alpha = xp.asarray(alphas, dtype=xp.float64)
        if spectral:
            quotient = self.projected / (self.eigenvalues + alpha[:, None])
            result = (self.vectors @ quotient[:, :, None])[:, :, 0]
        else:
            system = self.gram.copy()
            index = xp.arange(self.width)
            system[:, index, index] += alpha[:, None]
            with linalg_guard():
                result = xp.linalg.solve(system, self.rhs[:, :, None])[:, :, 0]
        residual = (self.gram @ result[:, :, None])[:, :, 0] + alpha[:, None] * result - self.rhs
        relative = xp.sqrt(xp.sum(residual * residual, axis=1)) / xp.maximum(xp.sqrt(xp.sum(self.rhs * self.rhs, axis=1)), 1.0)
        if not bool(to_host(xp.isfinite(result).all())) or np.any(to_host(relative) > 1e-7):
            raise ValueError('Пакетное решение Ridge не прошло проверку невязки')
        return result

    def predictions(self, coefficients):
        train = xp.maximum((self.x @ coefficients[:, :, None])[:, :, 0] + self.y_mean[:, None], 0.0)
        query = xp.maximum((self.q @ coefficients[:, :, None])[:, :, 0] + self.y_mean[:, None], 0.0)
        return train, query

    def reduce(self, prediction, actual, mask):
        error = xp.where(mask, prediction - actual, 0.0)
        totals = xp.stack([mask.sum(axis=1), xp.where(mask, actual, 0.0).sum(axis=1),
                           xp.where(mask, prediction, 0.0).sum(axis=1), xp.abs(error).sum(axis=1), error.sum(axis=1)], axis=1)
        return [totals_to_metrics(values) for values in to_host(totals)]

    def metrics(self, coefficients, include_original=False):
        train, query = self.predictions(coefficients)
        result = dict(train=self.reduce(train, self.y, self.mask))
        if self.items[0]['phase'] != 'full':
            result['query'] = self.reduce(query, self.qy, self.query_mask)
        if include_original:
            result['original_train'] = self.reduce(train, self.y, self.original_mask)
        return result, to_host(query)

In [ ]:
def route_features(data, origins, r):
    raw = data.matrix(origins,r)
    names = data.cols+data.corr_names
    keep = [i for i,c in enumerate(names) if c not in ('dayofweek','is_weekend')]
    hours = times[origins] + data.h*STEP
    angle = (hours.hour*2+hours.minute/30)*2*np.pi/48
    matrix = np.column_stack([raw[:,keep],np.sin(angle),np.cos(angle)]).astype(np.float64)
    matrix[~np.isfinite(matrix)] = np.nan
    return matrix

def route_item(data, origins, query, r, stage):
    a = route_features(data,origins,r)
    b = route_features(data,query,r)
    imputer = SimpleImputer(strategy='median',keep_empty_features=True)
    a = imputer.fit_transform(a)
    b = imputer.transform(b)
    active = np.flatnonzero(np.ptp(a,axis=0)>0)
    if not len(active):
        active = [0]
    scaler = StandardScaler()
    a = scaler.fit_transform(a[:,active])
    b = scaler.transform(b[:,active])
    mean = a.mean(axis=0)
    y = target[origins+data.h,r].astype(np.float64)
    qy = np.zeros(len(query)) if stage=='full' else target[query+data.h,r].astype(np.float64)
    return dict(x=a-mean,query_x=b-mean,y=y,y_mean=float(y.mean()),query_y=qy,
                original_train=np.ones(len(y),dtype=bool),phase=stage,r=r)

In [ ]:
run_local()